# Phase 3: Predictive Modeling

Paper section 2.3.1 / Graph 2.1. One model per observation age (not one global
model), predicting `future_paid` from the claim's state at the snapshot.
Ages **4** (1 year) and **8** (2 years) -- the quarter-analog of the paper's
P(12)/P(24), chosen for coverage on both sides of the time split.

Linear regression first as a baseline, then LightGBM.

**Final model: LightGBM, the 8 features below, raw-dollar target, predictions
clipped at zero.** Age-4 test R² = 0.237. The ceiling test below shows the
attainable maximum is ~0.78, so this is a weak fit -- see "Limitations" at the
end for why, and why it may still be enough for Phases 4-5.

## Load data

`build_snapshot_triangle()` from `src/snapshot_builder.py` -- core mechanism only,
matches `02_snapshot_triangle_build.ipynb` exactly, no feature columns.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from snapshot_builder import build_snapshot_triangle, ceil_period, load_visible, assert_no_leakage, CUTOFF
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from lightgbm import LGBMRegressor


df = pd.read_csv('../data/synthetic_transactions_with_covariates.csv')
snapshot_triangle = build_snapshot_triangle(df, cutoff=CUTOFF, step=1)
snapshot_triangle.head()

portfolio: 3624 claims
excluded (notified at or after the last grid point, 40): 301 (8.3%) -- no room left to show any development


snapshot triangle: 382485 rows (379840 observed, 2645 settled-exit), 3323 claims represented


,claim_no,snapshot_period,observation_age,observation_period,row_status,future_paid,paid_to_date,is_settled_at_obs
0,1,2,1.0,3.0,observed,0.0,0.0,False
1,1,2,2.0,4.0,observed,0.0,0.0,False
2,1,2,3.0,5.0,observed,0.0,0.0,False
3,1,2,4.0,6.0,observed,0.0,0.0,False
4,1,2,5.0,7.0,observed,0.0,0.0,False


## Add features

The minimal triangle only has `claim_no`, `snapshot_period`, `paid_to_date`, etc. --
no covariates, no payment-history detail. This project demos the
*mechanism* (snapshot triangle -> ML -> clustering -> chain ladder), not a
feature-engineering exercise.

**Kept:**

| feature | axis | why |
|---|---|---|
| `injury_severity` | static | how inherently large this claim is (one-hot encode -- sev 6 has a *lower* effect than sev 1, see CLAUDE.md) |
| `legal_representation` | static | litigated claims develop longer, pay more |
| `notidel` | static | late-reported claims behave differently -- classic reserving predictor |
| `paid_to_date` | dynamic | scale -- already a native column on the core triangle, no extra computation needed |
| `periods_since_notification` | dynamic | how far into development we're standing |
| `periods_since_last_payment` | dynamic | still active vs winding down |
| `has_any_payment` | dynamic flag | the other dynamic features are undefined before the first payment -- this lets the model tell "no payments yet" apart from "small payments" |
| `payment_acceleration` | dynamic | Taylor's "unpredictable dynamic covariate" -- used as-of-snapshot, never forecast forward. This is the one column that most directly demonstrates the paper's central claim |


**Banned** (leakage -- never allowed regardless of relevance, per
`BANNED_FEATURES` in `src/snapshot_builder.py`): `claim_size`, `setldel`,
`settlement_period`, `outstanding_at_snapshot`, `ultimate_observed`,
`payment_inflated`, `is_settled_at_obs`. The last one is a real column already
sitting in `snapshot_triangle` (part of the core 8) -- easy to grab by accident
since it's just sitting there; excluded explicitly when we build `X` in Step 2.


In [2]:
visible = load_visible(df)
claims = df.drop_duplicates('claim_no').copy()
claims['notification_period'] = (claims['occurrence_time'] + claims['notidel']).apply(ceil_period)

static_cols = claims[['claim_no', 'injury_severity', 'legal_representation', 'notidel']].copy()
static_cols['notidel'] = static_cols['notidel'].round(3)

payments_by_claim = {
    claim_no: claim_payments[['payment_period', 'payment_size']].values
    for claim_no, claim_payments in visible.groupby('claim_no')
}
notif_by_claim = claims.set_index('claim_no')['notification_period']

# one dynamic-feature row per (claim_no, snapshot_period) -- shared across every
# observation_age at that snapshot, since these only depend on the claim's state
# as of the snapshot, not on how far ahead a given row is predicting
combos = snapshot_triangle[['claim_no', 'snapshot_period']].drop_duplicates()

dyn_rows = []
for claim_no, s in combos.itertuples(index=False):
    payments = payments_by_claim.get(claim_no, np.empty((0, 2)))

    def cum_paid(p, _payments=payments):
        return _payments[_payments[:, 0] <= p][:, 1].sum()

    paid_so_far = payments[payments[:, 0] <= s]
    n_payments_to_date = int(len(paid_so_far))  # only needed internally -- not kept as a feature
    periods_since_last_payment = (
        s - int(paid_so_far[:, 0].max()) if n_payments_to_date else None
    )
    # payment_acceleration: this period's payment vs. the one before it (a fixed
    # 1-period lookback, unrelated to the triangle's own grid spacing -- no
    # variable needed since it's never anything but 1)
    paid_last = cum_paid(s) - cum_paid(s - 1)
    paid_prior = cum_paid(s - 1) - cum_paid(s - 2)
    payment_acceleration = round(paid_last / paid_prior, 3) if paid_prior > 1e-6 else None

    dyn_rows.append(dict(
        claim_no=claim_no, snapshot_period=s,
        periods_since_notification=s - notif_by_claim[claim_no],
        periods_since_last_payment=periods_since_last_payment,
        payment_acceleration=payment_acceleration,
        has_any_payment=n_payments_to_date > 0,
    ))
dynamic_cols = pd.DataFrame(dyn_rows)

snapshot_triangle = snapshot_triangle.merge(static_cols, on='claim_no', how='left')
snapshot_triangle = snapshot_triangle.merge(dynamic_cols, on=['claim_no', 'snapshot_period'], how='left')
snapshot_triangle.head()

,claim_no,snapshot_period,observation_age,observation_period,row_status,future_paid,paid_to_date,is_settled_at_obs,injury_severity,legal_representation,notidel,periods_since_notification,periods_since_last_payment,payment_acceleration,has_any_payment
0,1,2,1.0,3.0,observed,0.0,0.0,False,1,Y,0.93,0,NaN,NaN,False
1,1,2,2.0,4.0,observed,0.0,0.0,False,1,Y,0.93,0,NaN,NaN,False
2,1,2,3.0,5.0,observed,0.0,0.0,False,1,Y,0.93,0,NaN,NaN,False
3,1,2,4.0,6.0,observed,0.0,0.0,False,1,Y,0.93,0,NaN,NaN,False
4,1,2,5.0,7.0,observed,0.0,0.0,False,1,Y,0.93,0,NaN,NaN,False


In [3]:
# confirm the finalized feature list is leak-free before it ever touches a model
FEATURE_COLUMNS = [
    'injury_severity', 'legal_representation', 'notidel', 'paid_to_date',
    'periods_since_notification', 'periods_since_last_payment',
    'has_any_payment', 'payment_acceleration',
]
assert_no_leakage(FEATURE_COLUMNS)
print(f'{len(FEATURE_COLUMNS)} features, no leakage:', FEATURE_COLUMNS)

8 features, no leakage: ['injury_severity', 'legal_representation', 'notidel', 'paid_to_date', 'periods_since_notification', 'periods_since_last_payment', 'has_any_payment', 'payment_acceleration']


## Setup

Only `row_status == "observed"` rows have a real `future_paid` target (settled
rows are `None` -- exit markers, not training data).

**The split is by time** (`snapshot_period <= 28` train / `> 28` test):
standing at a snapshot and predicting forward is the actual job, so this is the
only score reported anywhere below.

Everything is scored through one `evaluate()` function so every number in the
notebook is directly comparable. It always scores in **dollars** and always
clips predictions at zero -- both LightGBM and LinearRegression emit negative
predictions, and a claim cannot pay a negative amount.

`bias_pct` is total predicted vs total actual dollars. R² and MAE say nothing
about whether the reserve *adds up*, which is the thing Phase 5 depends on.

In [4]:
age4 = snapshot_triangle[
    (snapshot_triangle['observation_age'] == 4) &
    (snapshot_triangle['row_status'] == 'observed')]

# future_paid answers one specific question: "starting from this snapshot, how
# much more money gets paid over the next observation_age quarters". It's the
# target because it's the one number genuinely unknown as of the snapshot that
# only becomes knowable later -- exactly what a reserving model must forecast.
train_mask = age4['snapshot_period'] <= CUTOFF - 12
print(f'{len(age4)} age-4 rows -- {train_mask.sum()} train (snapshot<=28), '
      f'{(~train_mask).sum()} test (snapshot>28)')

categorical_cols = ['injury_severity', 'legal_representation']
# NaN here means "no payment yet" (has_any_payment already flags this) -- fill
# with 0 so LinearRegression can accept the column at all; the flag column
# tells the model when to trust it vs. ignore it
nan_prone_cols = ['periods_since_last_payment', 'payment_acceleration']


def make_preprocessor():
    # a fresh transformer per model -- Pipeline fits its steps in place rather
    # than cloning them, so sharing one instance lets each fit clobber the last
    return ColumnTransformer(
        transformers=[
            ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
            ('impute', SimpleImputer(strategy='constant', fill_value=0), nan_prone_cols),
        ],
        remainder='passthrough')


def lgbm(**kw):
    return LGBMRegressor(random_state=20200131, verbose=-1, **kw)


def score(name, actual_train, pred_train, actual_test, pred_test):
    """One scorer for every model in the notebook -- ML, chain ladder, or a flat
    constant -- so every number below is directly comparable.

    bias_pct is total predicted vs total actual dollars. R2 and MAE say nothing
    about whether the reserve *adds up*, which is what Phase 5 depends on."""
    return dict(model=name,
                train_r2=r2_score(actual_train, pred_train),
                test_r2=r2_score(actual_test, pred_test),
                test_mae=mean_absolute_error(actual_test, pred_test),
                bias_pct=(pred_test.sum() / actual_test.sum() - 1) * 100)


def evaluate(name, frame, cols, regressor=None, log_target=False):
    tr = frame['snapshot_period'] <= CUTOFF - 12
    y = frame['future_paid'].astype(float)

    pipe = Pipeline([('preprocessor', make_preprocessor()),
                     ('regressor', regressor if regressor is not None else lgbm())])
    pipe.fit(frame.loc[tr, cols], np.log1p(y[tr]) if log_target else y[tr])

    def predict(mask):
        p = pipe.predict(frame.loc[mask, cols])
        return np.clip(np.expm1(p) if log_target else p, 0, None)

    return score(name, y[tr], predict(tr), y[~tr], predict(~tr))


def show(rows):
    print(pd.DataFrame(rows).to_string(index=False, formatters={
        'train_r2': '{:.3f}'.format, 'test_r2': '{:.3f}'.format,
        'test_mae': '${:,.0f}'.format, 'bias_pct': '{:+.1f}%'.format}))

20784 age-4 rows -- 14951 train (snapshot<=28), 5833 test (snapshot>28)


## Baselines: chain ladder first, then the models

**Chain ladder is the bar that matters.** It's what reserving actually does
without any ML, and it's what Phase 5 compares against, so beating the
predict-the-average baseline proves nothing on its own.

The mechanic: fit a development factor `f` on the training snapshots only,

```
f = (total paid_to_date + total future_paid) / total paid_to_date
predicted future_paid = paid_to_date x (f - 1)
```

It's a ratio of sums rather than a mean of ratios -- the standard estimator, so
large claims carry proportionate weight instead of one tiny claim's freak ratio
swinging the factor. Two versions: a single global factor (the simplest form),
and factors that vary by how mature the claim is at the snapshot, which is
closer to what a real chain ladder does since development slows with age.

Its structural weakness shows up immediately: the prediction is
`paid_to_date x something`, so **a claim that has paid nothing yet is predicted
to pay nothing, forever**. Individual-claim ML has no such constraint -- it can
read severity, lawyer, and reporting delay on a claim with $0 paid. That is
precisely the gap the paper argues ML fills.

LightGBM builds decision trees (splits on thresholds like `paid_to_date > 5000`)
instead of fitting one global straight line, so a handful of giant claims can't
warp every other prediction the way they do with linear regression.

In [5]:
def chain_ladder(name, frame, by=None):
    """Volume-weighted development factor, fit on training snapshots only.
    `by` groups the factor by a maturity column instead of using one global
    number; unseen groups at test time fall back to the global factor."""
    m = frame['snapshot_period'] <= CUTOFF - 12
    train = frame[m]
    base = train['paid_to_date'].sum()
    global_f = (base + train['future_paid'].sum()) / base

    if by is None:
        f = pd.Series(global_f, index=frame.index)
    else:
        s = train.groupby(by)[['paid_to_date', 'future_paid']].sum()
        table = ((s['paid_to_date'] + s['future_paid']) / s['paid_to_date']
                 ).replace([np.inf, -np.inf], np.nan)
        f = frame[by].map(table).fillna(global_f)

    pred = (frame['paid_to_date'] * (f - 1)).clip(lower=0)
    y = frame['future_paid'].astype(float)
    return score(name, y[m], pred[m], y[~m], pred[~m])


y_train = age4.loc[train_mask, 'future_paid'].astype(float)
y_test = age4.loc[~train_mask, 'future_paid'].astype(float)
flat = lambda n: np.full(n, y_train.mean())

show([chain_ladder('chain ladder (one factor)', age4),
      chain_ladder('chain ladder (by maturity)', age4, by='periods_since_notification'),
      score('dumb baseline (train mean)', y_train, flat(len(y_train)), y_test, flat(len(y_test))),
      evaluate('LinearRegression', age4, FEATURE_COLUMNS, LinearRegression()),
      evaluate('LightGBM', age4, FEATURE_COLUMNS)])

zero_paid = (age4.loc[~train_mask, 'paid_to_date'] == 0).mean() * 100
print(f'\nchain ladder is multiplicative, so it predicts $0 for the '
      f'{zero_paid:.0f}% of test rows that have paid_to_date = $0')
print(f'\ntarget shape: {(y_test == 0).mean()*100:.1f}% of test rows are exactly $0, '
      f'median=${y_test.median():,.0f}, mean=${y_test.mean():,.0f}, max=${y_test.max():,.0f}')
print(f'              top 1% of rows carry '
      f'{y_test.nlargest(int(len(y_test)*0.01)).sum()/y_test.sum()*100:.0f}% of all dollars')

                     model train_r2 test_r2 test_mae bias_pct
 chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%
dumb baseline (train mean)    0.000  -0.004  $84,829   -27.0%
          LinearRegression    0.073   0.043  $88,880   -10.3%
                  LightGBM    0.755   0.237  $73,456    -6.5%

chain ladder is multiplicative, so it predicts $0 for the 23% of test rows that have paid_to_date = $0

target shape: 9.0% of test rows are exactly $0, median=$12,859, mean=$74,600, max=$9,306,051
              top 1% of rows carry 30% of all dollars


### Chain ladder gets the total right and the individuals wrong

```
                     model train_r2 test_r2 test_mae bias_pct
 chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%
dumb baseline (train mean)    0.000  -0.004  $84,829   -27.0%
          LinearRegression    0.073   0.043  $88,880   -10.3%
                  LightGBM    0.755   0.237  $73,456    -6.5%
```

Both chain ladders score **negative R²** -- worse than predicting the average
for every claim. That is not a bug, and it is not really a defeat either:
**chain ladder is an aggregate method being asked an individual question.**

Look at `bias_pct` instead, which is the job it was built for. The
by-maturity chain ladder lands at **-5.8%** on total dollars, effectively tied
with LightGBM's **-6.5%**, and both beat the predict-the-mean baseline's -27%.
So on the portfolio total the two methods agree; where they differ is
*allocation* -- deciding which claims those dollars belong to. LightGBM
explains 24% of the row-level variance, chain ladder explains none.

That split is the whole premise of the paper. Phase 4 clusters claims on their
predicted trajectories, and clustering needs claims **ranked and separated**,
not portfolio totals. Chain ladder cannot rank claims at all, because its
prediction is `paid_to_date x factor` -- so it predicts **$0 for the 23% of
test rows with nothing paid yet**, and simply rescales everyone else in the
same proportion. A claim with severity 6 and a lawyer looks identical to a
minor claim if both have paid the same amount so far.

Two honest caveats:

- Letting the factor vary with claim maturity matters a lot (-0.590 to -0.194).
  A single global factor across all maturities is a strawman; the by-maturity
  version is the fair bar.
- This applies chain ladder **row by row**, which is the right comparison for a
  per-claim model but is not how it is normally used. The proper aggregate
  comparison -- build a real triangle, apply chain ladder, compare against
  per-cluster triangles -- is Phase 5's job, and this result does not
  pre-empt it.

## Is R² = 0.24 a bad model, or a hard target?

Worth knowing before spending effort tuning. Give a model the answer's
ingredients -- the true final cost, the true amount still outstanding, the true
settlement date -- and see how high it can get. That's the ceiling: honest
features are at best noisy proxies for these, so nothing legitimate beats it.

**These columns are all on `BANNED_FEATURES` on purpose and none of this feeds
Phases 4-5.** Adding them one at a time also shows *which* unknown is worth
approximating.

(Note `periods_to_settlement` slips past `assert_no_leakage()` -- the guard
matches exact column names, so a renamed derivative of a banned column passes
straight through.)

In [6]:
claim_statics = claims.set_index('claim_no')
settlement_period = (claim_statics['occurrence_time'] + claim_statics['notidel']
                     + claim_statics['setldel']).apply(ceil_period)

# age4_diag = age4 + leaky diagnostic columns. `age4` itself stays clean, and
# only the explicitly-named lists below ever reach these.
age4_diag = age4.copy()
age4_diag['claim_size'] = age4_diag['claim_no'].map(claim_statics['claim_size'])
age4_diag['setldel'] = age4_diag['claim_no'].map(claim_statics['setldel'])
age4_diag['outstanding_at_snapshot'] = age4_diag['claim_size'] - age4_diag['paid_to_date']
age4_diag['periods_to_settlement'] = (
    age4_diag['claim_no'].map(settlement_period) - age4_diag['snapshot_period'])

ORACLE = ['outstanding_at_snapshot', 'periods_to_settlement', 'claim_size', 'setldel']

show([evaluate('honest (our 8 features)', age4_diag, FEATURE_COLUMNS),
      evaluate('+ true outstanding', age4_diag, FEATURE_COLUMNS + ['outstanding_at_snapshot']),
      evaluate('+ true settlement date', age4_diag, FEATURE_COLUMNS + ['periods_to_settlement']),
      evaluate('ORACLE (all four)', age4_diag, FEATURE_COLUMNS + ORACLE),
      # capacity check: if test R² barely moves, the default wasn't underfitting
      evaluate('ORACLE, 15x the trees', age4_diag, FEATURE_COLUMNS + ORACLE,
               lgbm(n_estimators=1500, learning_rate=0.03, num_leaves=63))])

                  model train_r2 test_r2 test_mae bias_pct
honest (our 8 features)    0.755   0.237  $73,456    -6.5%
     + true outstanding    0.826   0.297  $62,092    -1.1%
 + true settlement date    0.886   0.442  $46,887    -4.1%
      ORACLE (all four)    0.970   0.767  $16,404    -3.9%
  ORACLE, 15x the trees    0.994   0.784  $17,428    -2.6%


### The ceiling is ~0.78

So 0.24 is roughly **30% of what's attainable** -- a weak fit, not a hopeless
target. Two things follow:

**Timing matters ~3x more than size.** Knowing the settlement date alone is
worth about +0.21; knowing the true outstanding alone about +0.06. Together
they far exceed the sum of their parts, because `future_paid` is essentially
*outstanding × (share of the remaining lifetime these 4 quarters cover)* -- a
multiplicative relationship a tree can only exploit holding both halves.

**Capacity is not the constraint.** 15x the trees moves test R² by ~0.02 while
train R² climbs to 0.99 and test *MAE gets worse* -- the model is memorising,
not underfitting. Tuning is not where the missing 0.5 lives.

## Taming the tail: log target, then Tweedie

Payments run from $0 to $9.3M and the top 1% of rows carry ~30% of all dollars,
so squared-error loss spends most of its effort on a handful of giant claims.
Two standard fixes:

**Log target** -- fit on `log1p(dollars)` so the giants stop dominating, then
convert back with `expm1` to score in dollars.

**Tweedie** -- the same tail-taming idea without the back-transform: a log link
that still predicts dollars directly. It's the standard actuarial choice for
pure premium, motivated by a target with a **point mass at zero** plus a heavy positive
tail -- meaning a large pile of values sitting at *exactly* $0, not merely
small ones, which is what breaks the smooth-distribution assumption ordinary
regression makes. Worth checking that premise against this data: only **9%** of
age-4 rows are exactly zero, because rows only exist between notification and
settlement, so most windows do contain a payment. Both `power=1.3` and `1.5`
are tested rather than assumed.

In [7]:
show([evaluate('raw $ target', age4, FEATURE_COLUMNS),
      evaluate('log1p target', age4, FEATURE_COLUMNS, log_target=True),
      evaluate('tweedie, power=1.3', age4, FEATURE_COLUMNS,
               lgbm(objective='tweedie', tweedie_variance_power=1.3)),
      evaluate('tweedie, power=1.5', age4, FEATURE_COLUMNS,
               lgbm(objective='tweedie', tweedie_variance_power=1.5))])

             model train_r2 test_r2 test_mae bias_pct
      raw $ target    0.755   0.237  $73,456    -6.5%
      log1p target    0.185  -0.007  $68,783   -62.5%
tweedie, power=1.3    0.816   0.167  $68,170   -24.8%
tweedie, power=1.5    0.754   0.162  $68,220   -27.0%


### Both lose to the plain dollar target -- keep raw dollars

**The log target fails badly.** It improves MAE but drives R² to ~0 and
under-books total dollars by **~62%**. Both effects have the same cause:
`expm1(mean of logs)` recovers something like a geometric mean, so it predicts
the *typical* row well -- which is exactly what MAE rewards -- and quietly
ignores the tail, which is where the dollars are. A model that books 37 cents
on the dollar is unusable in Phase 5 no matter how good its MAE looks.

**Tweedie is the milder version of the same trade** and still under-books.
That fits the 9%-zeros finding above: without a real point mass at zero, the
distributional assumption Tweedie is built for doesn't hold here, so its
theoretical advantage doesn't materialise.

Raw dollars is the only option close to unbiased in total, and `bias_pct` is
the column that reveals it -- R² and MAE both prefer the wrong model here.

## Final model

LightGBM on the 8 features, raw-dollar target, clipped at zero. One model per
observation age, as the paper does -- age 4 and age 8, which Phase 4 needs as
the two points of each claim's predicted trajectory `[P(4), P(8)]`.

Both are trained on `snapshot_period <= 28` only, so the reported scores stay
honest. `predict_future_paid()` is the entry point Phase 4 calls.

In [8]:
FINAL_AGES = [4, 8]
final_models, scores = {}, []

for age in FINAL_AGES:
    rows = snapshot_triangle[(snapshot_triangle['observation_age'] == age) &
                             (snapshot_triangle['row_status'] == 'observed')]
    scores.append(evaluate(f'age {age} (n={len(rows):,})', rows, FEATURE_COLUMNS))

    tr = rows['snapshot_period'] <= CUTOFF - 12
    final_models[age] = Pipeline([
        ('preprocessor', make_preprocessor()), ('regressor', lgbm())
    ]).fit(rows.loc[tr, FEATURE_COLUMNS], rows.loc[tr, 'future_paid'].astype(float))

show(scores)


def predict_future_paid(age, frame):
    """Phase 4 entry point: dollars predicted to be paid in the next `age`
    quarters, for any frame carrying FEATURE_COLUMNS. Clipped at zero."""
    return np.clip(final_models[age].predict(frame[FEATURE_COLUMNS]), 0, None)

           model train_r2 test_r2 test_mae bias_pct
age 4 (n=20,784)    0.755   0.237  $73,456    -6.5%
age 8 (n=17,878)    0.802   0.256 $100,601    -7.7%


## Hyperparameter tuning -- quick check

LightGBM has been running on plain defaults the whole time
(`LGBMRegressor(random_state=20200131, verbose=-1)`). Tuning tries a few
different settings -- tree depth, learning speed, how many trees -- and keeps
whichever scores best.

**Picking the winner has to respect the same time rule as everything else,**
so it needs its own three-way split instead of reusing `evaluate()`'s
train/test: train on snapshot <= 20, pick the winner on snapshot 21-28, then
check it once, for real, on the untouched test set (snapshot > 28).

In [9]:
tune_train = age4['snapshot_period'] <= 20
tune_val = (age4['snapshot_period'] > 20) & (age4['snapshot_period'] <= CUTOFF - 12)
y_tune_val = age4.loc[tune_val, 'future_paid'].astype(float)

candidates = {
    'default': {},
    'fewer leaves (simpler)': {'num_leaves': 15},
    'slower learner': {'n_estimators': 300, 'learning_rate': 0.03},
}

tuning_rows = []
for name, params in candidates.items():
    pipe = Pipeline([('preprocessor', make_preprocessor()), ('regressor', lgbm(**params))])
    pipe.fit(age4.loc[tune_train, FEATURE_COLUMNS],
             age4.loc[tune_train, 'future_paid'].astype(float))
    pred = np.clip(pipe.predict(age4.loc[tune_val, FEATURE_COLUMNS]), 0, None)
    tuning_rows.append(dict(settings=name, val_r2=round(r2_score(y_tune_val, pred), 3)))

tuning_df = pd.DataFrame(tuning_rows).sort_values('val_r2', ascending=False)
print(tuning_df.to_string(index=False))

best_params = candidates[tuning_df.iloc[0]['settings']]
show([evaluate('LightGBM', age4, FEATURE_COLUMNS),
      evaluate('LightGBM (tuned)', age4, FEATURE_COLUMNS, lgbm(**best_params))])

              settings  val_r2
fewer leaves (simpler)   0.262
               default   0.260
        slower learner   0.256


           model train_r2 test_r2 test_mae bias_pct
        LightGBM    0.755   0.237  $73,456    -6.5%
LightGBM (tuned)    0.674   0.253  $74,229    -6.7%


## Limitations

**The fit is weak and the ceiling test says most of the gap is real.** 0.24
against an attainable ~0.78. The missing piece is knowing *when* a claim
settles, and settlement timing in this data is driven by `setldel`, which the
covariates barely determine. Closing that gap means a duration/survival model
feeding the payment model -- a substantially larger build than this scoping
project covers, and the kind of thing that needs real data to justify.

**What was tried and rejected**, so it isn't repeated: log and Tweedie targets
(both under-book total dollars); more model capacity (memorises, test MAE gets
worse); an estimated-ultimate feature built by predicting `claim_size` from the
covariates (that model only reached R² = 0.08, so the feature added noise, not
information -- SynthETIC draws claim size as a lognormal *within* each covariate
cell, and that spread dwarfs the between-cell signal); and splitting by claim
instead of by time, which scored *lower* and so ruled out the time split as the
cause.

**R² is a noisy metric on this target.** Across claim-split folds it ranged from
-0.27 to +0.36 -- one fold catching a few mega-claims is enough to score worse
than predicting the mean. Small R² differences here are noise.

**Hyperparameter tuning was tried and didn't help.** A validation slice inside
the training period picked the best of a few settings; test R² moved from 0.237
to 0.253 -- a small nudge, not a fix. Consistent with the ceiling test: the gap is about
missing *information* (true settlement timing), not model configuration.

**Still untested, and cheap if revisited:** `n_payments_to_date` (computed in
the feature cell and currently thrown away), and dollars paid in the last 4
quarters.

**For the write-up:** these numbers describe *this synthetic dataset*, not
individual claim reserving in general. SynthETIC's payment timing is driven by
`setldel`, which the covariates barely determine, so the covariate-to-
development link this method relies on is deliberately weak here in a way real
claim data may not be.